# 02 — Data Quality Dimensions

**Difficulty:** ⭐⭐ | **Estimated Time:** 2 hours  
**Week 4 · Data Fundamentals & Preprocessing Pipelines**

---

## Why Does This Matter for ML?

The single most common reason ML models fail in production is not a bad algorithm — it's bad data. A neural network with a billion parameters cannot compensate for data that is missing, inconsistent, inaccurate, or stale. In the fraud detection domain, poor data quality has real consequences: genuine fraud goes undetected (financial loss) or legitimate transactions get blocked (customer churn).

**Garbage in, garbage out** is not just a saying — it is the #1 root cause of ML project failures cited in industry surveys.

After this notebook you will be able to:
- Measure all four dimensions of data quality programmatically
- Detect and flag quality problems before they corrupt your model
- Build a reusable `DataQualityReport` class for any dataset
- Quantify how much quality improves after cleaning

---

## Real-World Analogy: Building on a Cracked Foundation

Imagine hiring the world's best architect to design a skyscraper, but the construction crew lays the foundation on cracked, uneven ground. No matter how brilliant the architectural plans are, the building will lean, crack, and eventually fail.

Your ML model is the skyscraper. Your data is the foundation. No algorithm — no matter how sophisticated — can save a model built on data that is:
- Full of holes (**incomplete**)
- Contradicting itself (**inconsistent**)
- Factually wrong (**inaccurate**)
- Out of date (**not timely**)

---

## The Four Dimensions of Data Quality

| Dimension | Plain English Question | Fraud Detection Example |
|---|---|---|
| **Completeness** | Is anything missing? | `merchant_category` is NULL for 15% of transactions |
| **Consistency** | Is the same thing stored the same way? | 'NYC', 'New York', 'NY' all mean the same city |
| **Accuracy** | Are values correct (not just present)? | `amount = -500` or `age = 150` |
| **Timeliness** | Is the data fresh enough to be useful? | Using 2019 spending patterns to detect 2024 fraud |

---

**Our running theme:** Credit card fraud detection — a messy, real-world dataset with all four types of quality problems injected synthetically.

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import datetime
import warnings
warnings.filterwarnings('ignore')

# Visual configuration
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ANSI color codes — used in the DataQualityReport for colored terminal output
# These codes are standard escape sequences that most terminals support
RED    = '\033[91m'   # bright red — for critical/fail
YELLOW = '\033[93m'   # bright yellow — for warning
GREEN  = '\033[92m'   # bright green — for pass/good
BOLD   = '\033[1m'    # bold text
RESET  = '\033[0m'    # reset all formatting
CYAN   = '\033[96m'   # bright cyan — for headers

print('Environment ready.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---

## Step 1 — Generate a Messy Dataset

Real-world data is never clean. We'll deliberately inject quality problems into our synthetic credit card dataset so we have concrete examples to measure and fix.

### Problems we'll inject:
- **15% missing values** in multiple columns (simulates NULL values from database joins failing)
- **Inconsistent categories** — `'FOOD'`, `'food'`, `'Food & Dining'` all meaning the same thing
- **Impossible values** — negative transaction amounts, age = 999
- **Stale records** — some transactions timestamped years in the past

In [ ]:
# ============================================================
# GENERATE MESSY SYNTHETIC CREDIT CARD TRANSACTION DATASET
# ============================================================

N = 5_000  # 5,000 transactions — manageable for visualization
np.random.seed(RANDOM_SEED)

# --- Base clean data (what an ideal data collection would look like) ---

base_date = datetime.datetime(2023, 1, 1)
random_seconds = np.random.randint(0, 365 * 24 * 3600, size=N)
timestamps = [base_date + datetime.timedelta(seconds=int(s)) for s in random_seconds]

amounts = np.round(np.random.lognormal(mean=4.0, sigma=1.1, size=N), 2)
amounts = np.clip(amounts, 1.0, 12000.0)

# Clean merchant categories (canonical form)
clean_categories = [
    'grocery', 'restaurant', 'gas_station', 'online_retail',
    'travel', 'entertainment', 'healthcare', 'atm_withdrawal'
]
cat_weights = [0.25, 0.20, 0.15, 0.18, 0.08, 0.06, 0.05, 0.03]
merchant_categories = np.random.choice(clean_categories, size=N, p=cat_weights)

user_ages = np.random.randint(18, 80, size=N).astype(float)  # float to allow NaN later
user_ids  = [f'USR{str(i).zfill(4)}' for i in np.random.randint(1, 501, size=N)]
is_fraud  = np.random.binomial(n=1, p=0.02, size=N)

# Assemble clean DataFrame
df_clean = pd.DataFrame({
    'transaction_id':    [f'TXN{str(i).zfill(5)}' for i in range(1, N + 1)],
    'timestamp':         timestamps,
    'amount':            amounts,
    'merchant_category': merchant_categories,
    'user_id':           user_ids,
    'user_age':          user_ages,
    'is_fraud':          is_fraud
})

print(f'Clean base dataset: {df_clean.shape}')
print('Columns:', list(df_clean.columns))
df_clean.head(3)

In [ ]:
# ============================================================
# INJECT QUALITY PROBLEMS into a copy of the clean dataset
# ============================================================
# We keep df_clean unchanged — it is our "ground truth" for comparison.
# df_messy is what we'd receive from a real, imperfect data pipeline.

df_messy = df_clean.copy()

# ---- PROBLEM 1: Missing Values (Completeness violations) ----
# Simulate NULL values that occur when database JOIN fails to match records
MISSING_RATE = 0.15  # 15% of values in selected columns go missing

for col, rate in [('merchant_category', 0.15), ('user_age', 0.20), ('amount', 0.05)]:
    # Randomly select rows to set as NaN
    n_missing = int(N * rate)
    missing_idx = np.random.choice(df_messy.index, size=n_missing, replace=False)
    df_messy.loc[missing_idx, col] = np.nan  # NaN is pandas' representation of missing

# ---- PROBLEM 2: Inconsistent Categories (Consistency violations) ----
# Real data often has the same concept entered different ways by different systems
# Map: grocery → multiple messy variants with different capitalization/spelling
inconsistency_map = {
    'grocery':        ['Grocery', 'GROCERY', 'groceries', 'Grocery Store', 'grocery'],
    'restaurant':     ['Restaurant', 'RESTAURANT', 'food & dining', 'Food & Dining', 'restaurant'],
    'gas_station':    ['Gas Station', 'GAS', 'gas_station', 'Fuel', 'gas'],
    'online_retail':  ['Online Retail', 'ONLINE', 'e-commerce', 'Online Shopping', 'online_retail'],
}

# For each category that has messy variants, randomly replace ~40% of its occurrences
for clean_cat, messy_variants in inconsistency_map.items():
    mask = (df_messy['merchant_category'] == clean_cat) & (~df_messy['merchant_category'].isna())
    cat_indices = df_messy.index[mask].tolist()
    n_to_corrupt = int(len(cat_indices) * 0.40)  # corrupt 40% of this category's entries
    corrupt_idx = np.random.choice(cat_indices, size=n_to_corrupt, replace=False)
    # Assign a random messy variant (excluding the clean form) to each corrupted row
    df_messy.loc[corrupt_idx, 'merchant_category'] = np.random.choice(
        [v for v in messy_variants if v != clean_cat], size=n_to_corrupt
    )

# ---- PROBLEM 3: Impossible Values (Accuracy violations) ----
# Inject values that violate domain constraints
n_bad = 80  # number of clearly impossible records to inject

# Negative transaction amounts (physically impossible: a transaction amount < 0)
neg_idx = np.random.choice(df_messy.index, size=30, replace=False)
df_messy.loc[neg_idx, 'amount'] = -np.abs(np.random.uniform(50, 2000, size=30)).round(2)

# Extreme/impossible ages (age = 999, 0, 200 — clearly data entry errors)
age_idx = np.random.choice(df_messy.index, size=25, replace=False)
df_messy.loc[age_idx, 'user_age'] = np.random.choice([0, 150, 200, 999, -5], size=25)

# Extreme transaction amounts beyond any realistic threshold
extreme_idx = np.random.choice(df_messy.index, size=25, replace=False)
df_messy.loc[extreme_idx, 'amount'] = np.random.uniform(50_000, 999_999, size=25).round(2)

# ---- PROBLEM 4: Stale Timestamps (Timeliness violations) ----
# Replace some recent 2023 timestamps with very old dates (2015-2018)
stale_idx = np.random.choice(df_messy.index, size=200, replace=False)
old_base = datetime.datetime(2015, 1, 1)
old_seconds = np.random.randint(0, 3 * 365 * 24 * 3600, size=200)  # 3 years of old data
df_messy.loc[stale_idx, 'timestamp'] = [
    old_base + datetime.timedelta(seconds=int(s)) for s in old_seconds
]

print(f'Messy dataset created: {df_messy.shape}')
print(f'Missing values injected:')
print(df_messy.isnull().sum().to_string())
print(f'\nSample of merchant_category values (messy):')
print(df_messy['merchant_category'].dropna().value_counts().head(12).to_string())

---

## Dimension 1 — Completeness

### What Is It?
**Plain English:** Completeness measures whether all the data you expected to have is actually there. A cell that should have a value but doesn't is called a **missing value** (or NULL, NaN, None depending on the system).

### Why Does It Matter for Fraud Detection?
- A transaction with no `merchant_category` cannot be categorized by risk level
- A user with no `user_age` cannot be matched against age-based fraud profiles
- Missing data forces every ML algorithm to make an assumption — and the wrong assumption corrupts predictions

### Types of Missingness
| Type | Plain English | Example |
|---|---|---|
| **MCAR** — Missing Completely At Random | Data missing due to random technical failure | Server crashed during logging |
| **MAR** — Missing At Random | Missingness depends on other observed columns | International cards have no zip code |
| **MNAR** — Missing Not At Random | Missingness depends on the missing value itself | High earners refuse to report income |

**MNAR is the most dangerous** because the missingness itself carries information that your model will miss.

### Completeness Score
$$\text{Completeness}(\%) = \frac{\text{Non-null cells}}{\text{Total cells}} \times 100$$

In [ ]:
# ============================================================
# DIMENSION 1: COMPLETENESS — Measure and Visualize Missing Data
# ============================================================

# Compute per-column missing statistics
# isnull() returns True where value is NaN, sum() counts the Trues
missing_count = df_messy.isnull().sum()
missing_pct   = df_messy.isnull().sum() / len(df_messy) * 100  # percentage missing per column
present_pct   = 100 - missing_pct  # percentage present (= completeness)

completeness_df = pd.DataFrame({
    'column':        missing_count.index,
    'missing_count': missing_count.values,
    'missing_pct':   missing_pct.values.round(2),
    'present_pct':   present_pct.values.round(2)
})
completeness_df = completeness_df.sort_values('missing_pct', ascending=False)

# Overall dataset completeness score (across ALL cells, not just per-column)
total_cells    = df_messy.shape[0] * df_messy.shape[1]  # rows × columns = total data points
missing_cells  = df_messy.isnull().sum().sum()           # total missing across all columns
overall_completeness = (total_cells - missing_cells) / total_cells * 100

print('=== COMPLETENESS REPORT ===')
print(completeness_df.to_string(index=False))
print(f'\nOverall completeness score: {overall_completeness:.1f}%')
print(f'Total cells: {total_cells:,}  |  Missing: {missing_cells:,}  |  Present: {total_cells - missing_cells:,}')

# Identify columns with critical missing rates (>10% is typically a red flag)
critical_cols = completeness_df[completeness_df['missing_pct'] > 10]['column'].tolist()
print(f'\nCRITICAL (>10% missing): {critical_cols}')
print('These columns need investigation: is the data MCAR, MAR, or MNAR?')

In [ ]:
# ============================================================
# VISUALIZE COMPLETENESS: Two complementary views
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dimension 1: Data Completeness Analysis\nCredit Card Transaction Dataset',
             fontsize=13, fontweight='bold')

# --- Left: Horizontal bar chart of missing % per column ---
ax = axes[0]
# Color bars based on severity: >10% = red, >5% = orange, else = green
bar_colors = [
    '#E74C3C' if p > 10 else '#F39C12' if p > 5 else '#27AE60'
    for p in completeness_df['missing_pct']
]
bars = ax.barh(completeness_df['column'], completeness_df['missing_pct'],
               color=bar_colors, edgecolor='white', linewidth=1.2, height=0.55)

# Annotate each bar with its percentage
for bar, pct in zip(bars, completeness_df['missing_pct']):
    if pct > 0:
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)

# Add a threshold reference line
ax.axvline(10, color='#E74C3C', linestyle='--', lw=1.5, alpha=0.7, label='10% threshold')
ax.set_xlabel('% Missing Values')
ax.set_title('Missing Rate per Column')
ax.legend(fontsize=8)

# Add a legend for bar colors
legend_patches = [
    mpatches.Patch(color='#E74C3C', label='Critical (>10%)'),
    mpatches.Patch(color='#F39C12', label='Warning (5–10%)'),
    mpatches.Patch(color='#27AE60', label='OK (<5%)')
]
ax.legend(handles=legend_patches, fontsize=8, loc='lower right')

# --- Right: Missing value heatmap (missingno-style, using seaborn) ---
# Sample 100 rows for the heatmap — showing full 5,000 would be unreadable
ax2 = axes[1]
sample_100 = df_messy.sample(100, random_state=RANDOM_SEED)
# Create a boolean mask: True = missing (white), False = present (blue)
missing_mask = sample_100.isnull().astype(int)  # 1 = missing, 0 = present

sns.heatmap(
    missing_mask.T,          # transpose: columns as rows, samples as columns
    cmap=['#2ECC71', '#E74C3C'],  # green = present, red = missing
    cbar=False,               # no color bar needed — it's binary
    ax=ax2,
    linewidths=0,
    xticklabels=False         # hide 100 row numbers — too cluttered
)
ax2.set_title('Missing Value Pattern\n(100-row sample, red = missing)')
ax2.set_xlabel('Transactions (100 sample rows)')
ax2.set_ylabel('Column')

plt.tight_layout()
plt.show()

---

## Dimension 2 — Consistency

### What Is It?
**Plain English:** Consistency means the same real-world thing is always recorded the same way. When different systems, people, or time periods record the same concept differently, you have a consistency problem.

### Common Consistency Violations in Fraud Data

| Problem | Example | Why It Matters |
|---|---|---|
| **Case inconsistency** | 'GROCERY', 'Grocery', 'grocery' | Model treats them as 3 different categories |
| **Abbreviation variants** | 'NYC', 'New York', 'N.Y.C.' | Grouping and joins fail silently |
| **Type inconsistency** | amount = '1,234.50' (string) vs 1234.50 (float) | Cannot do math on strings |
| **Conflicting records** | User age=25 in transactions, age=45 in CRM | Which do you trust? |

### How to Detect Consistency Issues
The primary tool is **value counts**: look at the unique values in a categorical column and spot variants that clearly mean the same thing.

In [ ]:
# ============================================================
# DIMENSION 2: CONSISTENCY — Detect and Fix Category Variants
# ============================================================

# Step 1: Expose the inconsistency by looking at value counts
# dropna=False includes NaN in the count so we see the full picture
category_counts = df_messy['merchant_category'].value_counts(dropna=False)

print('=== RAW merchant_category VALUE COUNTS (messy) ===')
print(f'Total unique values: {df_messy["merchant_category"].nunique(dropna=True)}')
print(category_counts.to_string())
print(f'\nExpected: {len(["grocery","restaurant","gas_station","online_retail","travel","entertainment","healthcare","atm_withdrawal"])} canonical categories')
print(f'Found: {df_messy["merchant_category"].nunique()} unique strings — clearly inconsistent!')

In [ ]:
# ============================================================
# FIX CONSISTENCY: Standardize to canonical category names
# ============================================================
# The fix: create a mapping from all known variants → canonical form,
# then apply it. This is the standard approach in data cleaning.

# Define the canonical (correct, standardized) form for each variant
# Every messy variant maps to one clean value
category_mapping = {
    # Grocery variants
    'grocery':       'grocery',
    'Grocery':       'grocery',
    'GROCERY':       'grocery',
    'groceries':     'grocery',
    'Grocery Store': 'grocery',

    # Restaurant variants
    'restaurant':    'restaurant',
    'Restaurant':    'restaurant',
    'RESTAURANT':    'restaurant',
    'food & dining': 'restaurant',
    'Food & Dining': 'restaurant',

    # Gas station variants
    'gas_station':   'gas_station',
    'Gas Station':   'gas_station',
    'GAS':           'gas_station',
    'Fuel':          'gas_station',
    'gas':           'gas_station',

    # Online retail variants
    'online_retail':    'online_retail',
    'Online Retail':    'online_retail',
    'ONLINE':           'online_retail',
    'e-commerce':       'online_retail',
    'Online Shopping':  'online_retail',

    # Unchanged categories (no variants injected for these)
    'travel':         'travel',
    'entertainment':  'entertainment',
    'healthcare':     'healthcare',
    'atm_withdrawal': 'atm_withdrawal',
}

# Apply the mapping: .map() replaces each value with its mapped equivalent
# Values not in the mapping (like NaN) remain unchanged — we don't create fake data
df_messy['merchant_category_clean'] = df_messy['merchant_category'].map(category_mapping)

# Verify the standardization worked
print('=== AFTER STANDARDIZATION ===')
clean_counts = df_messy['merchant_category_clean'].value_counts(dropna=False)
print(clean_counts.to_string())

# Count how many rows were actually changed
changed = (df_messy['merchant_category'].fillna('__MISSING__') !=
           df_messy['merchant_category_clean'].fillna('__MISSING__')).sum()
print(f'\nRows standardized: {changed:,}')
print(f'Unique categories: {df_messy["merchant_category_clean"].nunique()} (expected: 8)')

In [ ]:
# ============================================================
# VISUALIZE CONSISTENCY: Before vs After standardization
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dimension 2: Consistency — Merchant Category Standardization',
             fontsize=13, fontweight='bold')

# --- Left: BEFORE — messy value counts ---
ax1 = axes[0]
before_counts = df_messy['merchant_category'].value_counts(dropna=False).head(16)
colors_before = ['#E74C3C' if v not in clean_categories else '#27AE60'
                 for v in before_counts.index.astype(str)]
ax1.barh(before_counts.index.astype(str), before_counts.values,
         color=colors_before, edgecolor='white', linewidth=1)
ax1.set_xlabel('Count')
ax1.set_title(f'BEFORE: {df_messy["merchant_category"].nunique()} unique values\n(red = inconsistent variant)')
ax1.invert_yaxis()  # most common at the top

# --- Right: AFTER — clean value counts ---
ax2 = axes[1]
after_counts = df_messy['merchant_category_clean'].value_counts(dropna=False)
ax2.barh(after_counts.index.astype(str), after_counts.values,
         color='#27AE60', edgecolor='white', linewidth=1)
# Annotate bars with counts
for i, (idx, count) in enumerate(after_counts.items()):
    ax2.text(count + 5, i, f'{count:,}', va='center', fontsize=9)
ax2.set_xlabel('Count')
ax2.set_title(f'AFTER: {df_messy["merchant_category_clean"].nunique()} canonical values\n(all variants merged)')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print('Consistency fix: reduced from many variants to 8 canonical categories.')
print('This allows correct grouping, encoding, and fraud-rate analysis per category.')

---

## Dimension 3 — Accuracy

### What Is It?
**Plain English:** Accuracy means values are present AND correct. A column can be fully complete (no NaN) but still inaccurate if the values are wrong.

Completeness answers: *Is the data there?*  
Accuracy answers: *Is it right?*

### Types of Accuracy Violations
| Type | Example | Detection Method |
|---|---|---|
| **Domain violations** | `amount = -500` (impossible) | Rule-based: amount must be > 0 |
| **Range violations** | `age = 150` (implausible) | Rule-based: 0 < age < 120 |
| **Statistical outliers** | Amount 100x above typical | Z-score > 5 from mean |
| **Logical contradictions** | Transaction at 9am in London AND 9am in New York | Cross-column consistency |
| **Pattern violations** | Card number doesn't match Luhn algorithm | Regex/checksum validation |

### Real Banking Rules for Accuracy
- Transaction amounts must be positive (negative = returns, tracked separately)
- No transaction can exceed the card's credit limit
- Transactions > \$10,000 trigger mandatory reporting under US Bank Secrecy Act
- A cardholder cannot be simultaneously in two geographically distant locations

### Statistical Accuracy: The Z-Score Method
**Plain English:** A z-score measures how many standard deviations a value is from the mean. A z-score of 5 means the value is 5 standard deviations away — if data is roughly normal, this happens by chance only 0.00003% of the time. Values this extreme are almost certainly errors.

In [ ]:
# ============================================================
# DIMENSION 3: ACCURACY — Rule-Based and Statistical Validation
# ============================================================

# --- Define validation rules as a dictionary ---
# Each rule: (column, condition_function, description)
# The condition returns True for VALID rows, False for VIOLATIONS

validation_rules = [
    {
        'rule_id':    'AMOUNT_POSITIVE',
        'column':     'amount',
        'check':      lambda df: df['amount'] > 0,  # amount must be positive
        'severity':   'CRITICAL',
        'description': 'Transaction amount must be positive (>0)'
    },
    {
        'rule_id':    'AMOUNT_REALISTIC',
        'column':     'amount',
        'check':      lambda df: df['amount'] <= 25_000,  # above $25k is implausible for retail
        'severity':   'WARNING',
        'description': 'Transaction amount must be <= $25,000 (above = likely data error or ATM fraud)'
    },
    {
        'rule_id':    'AGE_VALID',
        'column':     'user_age',
        'check':      lambda df: (df['user_age'] >= 18) & (df['user_age'] <= 110),  # plausible human age
        'severity':   'CRITICAL',
        'description': 'User age must be between 18 and 110'
    },
    {
        'rule_id':    'FRAUD_BINARY',
        'column':     'is_fraud',
        'check':      lambda df: df['is_fraud'].isin([0, 1]),  # must be exactly 0 or 1
        'severity':   'CRITICAL',
        'description': 'is_fraud label must be 0 or 1 (binary)'
    },
]

# --- Run all validation rules and collect results ---
accuracy_results = []

for rule in validation_rules:
    # Rows where the column is not null (we can only check accuracy for present values)
    not_null_mask = df_messy[rule['column']].notna()
    rows_checkable = not_null_mask.sum()  # number of non-null rows we can actually validate
    
    # Apply the validation check function; violations are where check = False
    valid_mask    = rule['check'](df_messy) & not_null_mask  # valid AND present
    violation_mask = (~rule['check'](df_messy)) & not_null_mask  # invalid AND present
    
    n_violations = violation_mask.sum()
    violation_pct = n_violations / rows_checkable * 100 if rows_checkable > 0 else 0
    
    accuracy_results.append({
        'rule_id':       rule['rule_id'],
        'severity':      rule['severity'],
        'rows_checked':  rows_checkable,
        'violations':    n_violations,
        'violation_pct': round(violation_pct, 2),
        'description':   rule['description']
    })

accuracy_df = pd.DataFrame(accuracy_results)

print('=== ACCURACY VALIDATION REPORT ===')
for _, row in accuracy_df.iterrows():
    status = 'FAIL' if row['violations'] > 0 else 'PASS'
    sev    = f"[{row['severity']}]" if row['violations'] > 0 else '[OK]     '
    print(f'{sev} {row["rule_id"]:<22} | {row["violations"]:>4} violations ({row["violation_pct"]:>5.1f}%) | {row["description"][:55]}')

total_violations = accuracy_df['violations'].sum()
print(f'\nTotal accuracy violations: {total_violations:,}')

In [ ]:
# ============================================================
# STATISTICAL ACCURACY: Z-score outlier detection on amount
# ============================================================
# Z-score tells us how unusual a value is relative to the distribution.
# Formula: z = (x - mean) / std
# |z| > 5 means the value is 5 standard deviations from the mean — almost certainly wrong.

# Work only on valid (positive) amounts to compute a meaningful z-score baseline
valid_amounts = df_messy['amount'].dropna()
valid_amounts = valid_amounts[valid_amounts > 0]  # exclude impossible negatives from baseline

amount_mean = valid_amounts.mean()
amount_std  = valid_amounts.std()

# Compute z-score for every amount (including nulls — they'll be NaN in z_scores)
df_messy['amount_zscore'] = (df_messy['amount'] - amount_mean) / amount_std

# Flag statistical outliers: |z| > 5 is almost certainly an error
stat_outliers = df_messy[np.abs(df_messy['amount_zscore']) > 5]
rule_violations = df_messy[df_messy['amount'] < 0]  # rule-based violations (negative)

print(f'Amount distribution (valid positives):')
print(f'  Mean:   ${amount_mean:>10,.2f}')
print(f'  Std:    ${amount_std:>10,.2f}')
print(f'  Median: ${valid_amounts.median():>10,.2f}')
print(f'\nStatistical outliers (|z-score| > 5): {len(stat_outliers)} records')
print(f'Rule-based violations (amount < 0):   {len(rule_violations)} records')

# Visualize the amount distribution with outlier flags
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dimension 3: Accuracy — Transaction Amount Violations', fontsize=13, fontweight='bold')

# Left: Histogram of amounts (log scale to see the distribution + outliers)
ax1 = axes[0]
# Clip at 5000 for visualization; extreme outliers would compress the visible range
plot_amounts = df_messy['amount'].dropna().clip(upper=5000)
ax1.hist(plot_amounts, bins=60, color='#4A90D9', alpha=0.7, edgecolor='white', linewidth=0.5)
ax1.axvline(0, color='#E74C3C', linestyle='--', lw=2, label='Zero threshold (amount must be > 0)')
ax1.set_xlabel('Transaction Amount ($)')
ax1.set_ylabel('Frequency')
ax1.set_title('Amount Distribution\n(clipped at $5,000 for visibility)')
ax1.legend(fontsize=8)

# Right: Z-score scatter plot
ax2 = axes[1]
# Sample 500 points for clarity; all points would overplot
sample_z = df_messy[['amount', 'amount_zscore']].dropna().sample(500, random_state=RANDOM_SEED)
# Color: red if |z| > 5, blue otherwise
point_colors = ['#E74C3C' if abs(z) > 5 else '#4A90D9' for z in sample_z['amount_zscore']]
ax2.scatter(range(len(sample_z)), sample_z['amount'], c=point_colors, alpha=0.6, s=15)
# Horizontal lines at the outlier threshold
threshold_high = amount_mean + 5 * amount_std
ax2.axhline(threshold_high, color='#E74C3C', linestyle='--', lw=1.5,
            label=f'5σ threshold (${threshold_high:,.0f})')
ax2.axhline(0, color='#E74C3C', linestyle=':', lw=1.5, label='Zero (must be positive)')
ax2.set_xlabel('Transaction index (500 sample)')
ax2.set_ylabel('Amount ($)')
ax2.set_title('Z-score Outlier Detection\n(red = statistical anomaly |z| > 5)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

---

## Dimension 4 — Timeliness

### What Is It?
**Plain English:** Timeliness measures whether the data is fresh enough to be useful for the task at hand. Old data can be internally consistent and accurate — but if the world has changed since it was collected, it no longer reflects current reality.

### Why Timeliness Is Critical for Fraud Detection

Fraud patterns are constantly evolving:
- **2019**: Card-present skimming at ATMs was dominant
- **2020**: COVID drove a massive spike in card-not-present (online) fraud
- **2022**: Buy-now-pay-later fraud emerged as a new attack vector
- **2024**: Synthetic identity fraud (AI-generated personas) dominates

A model trained on 2019 data would be completely blind to 2024 attack patterns. This is called **concept drift** — the statistical relationship between features and the target changes over time.

### Timeliness Metrics
| Metric | Formula | Good Threshold |
|---|---|---|
| **Record age** | Today − record.timestamp | < 90 days for fraud models |
| **Data freshness ratio** | % records < 30 days old | > 60% for production fraud |
| **Time gap detection** | Find periods with no data | No gaps > 24 hours |

### Concept Drift
**Plain English:** Concept drift is when the rules of the game change. Imagine learning to play chess, then everyone switches to checkers — your chess skills (your model's learned patterns) no longer apply.

In [ ]:
# ============================================================
# DIMENSION 4: TIMELINESS — Measure Data Freshness
# ============================================================

# Compute record age: how many days old is each transaction?
reference_date = datetime.datetime(2024, 1, 1)  # simulated "today" for this exercise

df_messy['timestamp_dt']  = pd.to_datetime(df_messy['timestamp'])
df_messy['record_age_days'] = (reference_date - df_messy['timestamp_dt']).dt.days

# Flag stale records: older than 365 days relative to our reference date
# For a 2023 dataset evaluated in Jan 2024: anything from 2022 or before is stale
STALENESS_THRESHOLD_DAYS = 365  # records older than 1 year from reference date are stale
df_messy['is_stale'] = df_messy['record_age_days'] > STALENESS_THRESHOLD_DAYS

# Timeliness statistics
stale_count   = df_messy['is_stale'].sum()
stale_pct     = stale_count / len(df_messy) * 100
fresh_pct     = 100 - stale_pct
median_age    = df_messy['record_age_days'].median()
oldest_record = df_messy['timestamp_dt'].min()
newest_record = df_messy['timestamp_dt'].max()

print('=== TIMELINESS REPORT ===')
print(f'Reference date ("today"):   {reference_date.date()}')
print(f'Oldest record:              {oldest_record.date()}  ({(reference_date - oldest_record).days} days ago)')
print(f'Newest record:              {newest_record.date()}  ({(reference_date - newest_record).days} days ago)')
print(f'Median record age:          {median_age:.0f} days')
print(f'Staleness threshold:        >{STALENESS_THRESHOLD_DAYS} days')
print(f'Stale records:              {stale_count:,}  ({stale_pct:.1f}%)')
print(f'Fresh records:              {len(df_messy) - stale_count:,}  ({fresh_pct:.1f}%)')

# Timeliness score: percentage of records that are fresh
timeliness_score = fresh_pct
print(f'\nTimeliness score: {timeliness_score:.1f}%  (higher = more recent data)')
if timeliness_score < 80:
    print('WARNING: More than 20% of records are stale — model may suffer from concept drift!')

In [ ]:
# ============================================================
# VISUALIZE TIMELINESS: Data freshness histogram + time coverage
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Dimension 4: Timeliness — Data Freshness Analysis', fontsize=13, fontweight='bold')

# --- Top: Record age histogram ---
ax1 = axes[0]
# Clip at 1500 days for readability (the injected old records go back to 2015)
ages_plot = df_messy['record_age_days'].clip(upper=1500)

# Split into fresh and stale for separate coloring
fresh_ages = ages_plot[df_messy['record_age_days'] <= STALENESS_THRESHOLD_DAYS]
stale_ages = ages_plot[df_messy['record_age_days'] > STALENESS_THRESHOLD_DAYS]

ax1.hist(fresh_ages, bins=40, color='#27AE60', alpha=0.75, label=f'Fresh (≤{STALENESS_THRESHOLD_DAYS} days)')
ax1.hist(stale_ages, bins=20, color='#E74C3C', alpha=0.75, label=f'Stale (>{STALENESS_THRESHOLD_DAYS} days)')
ax1.axvline(STALENESS_THRESHOLD_DAYS, color='#E74C3C', linestyle='--', lw=2,
            label=f'Staleness threshold ({STALENESS_THRESHOLD_DAYS} days)')
ax1.set_xlabel('Record Age (days from reference date Jan 1, 2024)')
ax1.set_ylabel('Number of Records')
ax1.set_title(f'Data Freshness Distribution: {fresh_pct:.1f}% fresh, {stale_pct:.1f}% stale')
ax1.legend(fontsize=9)

# --- Bottom: Transaction volume by month (shows temporal coverage gaps) ---
ax2 = axes[1]
df_messy['month_period'] = df_messy['timestamp_dt'].dt.to_period('M')
monthly_volume = df_messy.groupby('month_period').size().reset_index(name='count')
monthly_volume['month_str'] = monthly_volume['month_period'].astype(str)

# Color bars: stale years in red, current year in blue
bar_colors_t = [
    '#E74C3C' if int(m[:4]) < 2023 else '#4A90D9'
    for m in monthly_volume['month_str']
]
ax2.bar(monthly_volume['month_str'], monthly_volume['count'],
        color=bar_colors_t, edgecolor='white', linewidth=0.8)
ax2.set_xlabel('Month')
ax2.set_ylabel('Transactions')
ax2.set_title('Transaction Volume by Month (red = stale pre-2023 data injected)')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add legend patches manually
stale_patch = mpatches.Patch(color='#E74C3C', label='Stale (pre-2023)')
fresh_patch = mpatches.Patch(color='#4A90D9', label='Current (2023)')
ax2.legend(handles=[fresh_patch, stale_patch], fontsize=9)

plt.tight_layout()
plt.show()

---

## Overall Data Quality Score

Now we combine all four dimensions into a single **Data Quality Score** — a weighted composite that gives an at-a-glance summary of dataset health.

### Scoring Formula
$$\text{DQ Score} = w_1 \cdot \text{Completeness} + w_2 \cdot \text{Consistency} + w_3 \cdot \text{Accuracy} + w_4 \cdot \text{Timeliness}$$

Weights can be adjusted based on business context. For fraud detection, accuracy is critical (an inaccurate fraud label costs money directly), so we weight it higher.

In [ ]:
# ============================================================
# DATA QUALITY REPORT CLASS
# ============================================================
# A reusable class that computes all 4 quality dimensions
# and produces a colored summary report.

class DataQualityReport:
    """
    Computes and summarizes data quality across four dimensions:
    Completeness, Consistency, Accuracy, and Timeliness.

    Parameters
    ----------
    df : pd.DataFrame — the dataset to evaluate
    name : str — a human-readable name for this dataset (for reporting)
    timestamp_col : str — column containing datetime values (for timeliness)
    reference_date : datetime — the "today" to measure freshness against
    staleness_days : int — records older than this (in days) are considered stale
    accuracy_rules : list of dicts — each dict has 'rule_id', 'column', 'check'
    """

    def __init__(self, df, name='Dataset', timestamp_col=None,
                 reference_date=None, staleness_days=365, accuracy_rules=None):
        self.df             = df.copy()       # never mutate the caller's DataFrame
        self.name           = name
        self.timestamp_col  = timestamp_col
        self.reference_date = reference_date or datetime.datetime.now()
        self.staleness_days = staleness_days
        self.accuracy_rules = accuracy_rules or []  # empty list = skip accuracy check
        self.scores         = {}              # stores computed dimension scores

    def _compute_completeness(self):
        """Score = % of non-null cells across the entire DataFrame."""
        total = self.df.shape[0] * self.df.shape[1]   # total number of cells
        missing = self.df.isnull().sum().sum()          # total missing cells
        score = (total - missing) / total * 100
        # Per-column missing rates for detail reporting
        per_col = (self.df.isnull().sum() / len(self.df) * 100).round(1)
        return score, per_col

    def _compute_consistency(self, cat_columns=None):
        """Score: approximate — penalizes columns with high cardinality variance.
        In practice, consistency requires domain-specific mapping logic.
        Here we use a heuristic: ratio of unique values to expected unique values.
        """
        # Simple heuristic: 100% if no duplicate-intent values detected
        # More robust implementations would need a canonical reference vocabulary
        score = 100.0  # placeholder — in a real project, add domain-specific logic
        return score

    def _compute_accuracy(self):
        """Score = % of rule-checked values that pass all validation rules."""
        if not self.accuracy_rules:
            return 100.0, []  # no rules defined — assume perfect accuracy

        total_checked, total_violations = 0, 0
        results = []
        for rule in self.accuracy_rules:
            not_null = self.df[rule['column']].notna()
            n_checked = not_null.sum()
            n_violations = ((~rule['check'](self.df)) & not_null).sum()
            total_checked    += n_checked
            total_violations += n_violations
            results.append((rule['rule_id'], n_violations, n_checked))

        score = (total_checked - total_violations) / total_checked * 100 if total_checked > 0 else 100.0
        return score, results

    def _compute_timeliness(self):
        """Score = % of records that are fresher than staleness_days."""
        if self.timestamp_col is None or self.timestamp_col not in self.df.columns:
            return 100.0  # cannot assess timeliness without a timestamp column

        ts = pd.to_datetime(self.df[self.timestamp_col], errors='coerce')
        ages = (self.reference_date - ts).dt.days
        fresh = (ages <= self.staleness_days).sum()
        score = fresh / len(self.df) * 100
        return score

    def compute(self, weights=None):
        """Run all four dimension checks and compute the weighted composite score."""
        # Default weights: accuracy and completeness weighted slightly higher
        weights = weights or {'completeness': 0.30, 'consistency': 0.20,
                               'accuracy': 0.35, 'timeliness': 0.15}

        self.completeness_score, self.per_col_missing = self._compute_completeness()
        self.consistency_score = self._compute_consistency()
        self.accuracy_score, self.accuracy_details = self._compute_accuracy()
        self.timeliness_score  = self._compute_timeliness()

        # Weighted composite: sum of (score × weight) for all 4 dimensions
        self.composite_score = (
            self.completeness_score * weights['completeness'] +
            self.consistency_score  * weights['consistency'] +
            self.accuracy_score     * weights['accuracy'] +
            self.timeliness_score   * weights['timeliness']
        )
        self.scores = {
            'Completeness': self.completeness_score,
            'Consistency':  self.consistency_score,
            'Accuracy':     self.accuracy_score,
            'Timeliness':   self.timeliness_score,
            'Composite':    self.composite_score
        }
        return self

    def summary(self):
        """Print a colored, formatted quality report to the terminal."""
        def grade(score):
            if score >= 90: return GREEN + 'PASS' + RESET
            if score >= 75: return YELLOW + 'WARN' + RESET
            return RED + 'FAIL' + RESET

        print(CYAN + BOLD + f'\n{'='*55}' + RESET)
        print(CYAN + BOLD + f'  DATA QUALITY REPORT: {self.name}' + RESET)
        print(CYAN + BOLD + f'  Dataset: {self.df.shape[0]:,} rows × {self.df.shape[1]} columns' + RESET)
        print(CYAN + BOLD + f'{'='*55}' + RESET)

        dim_results = [
            ('Completeness', self.completeness_score, '(% non-null cells)'),
            ('Consistency',  self.consistency_score,  '(heuristic; needs domain rules)'),
            ('Accuracy',     self.accuracy_score,     '(% passing all validation rules)'),
            ('Timeliness',   self.timeliness_score,   f'(% records < {self.staleness_days}d old)'),
        ]
        for dim, score, note in dim_results:
            print(f'  {grade(score)} {dim:<15} {score:>6.1f}%  {note}')

        print(BOLD + f'{'─'*55}' + RESET)
        comp_color = GREEN if self.composite_score >= 90 else YELLOW if self.composite_score >= 75 else RED
        print(BOLD + comp_color + f'  COMPOSITE SCORE     {self.composite_score:>6.1f}%  (weighted)' + RESET)
        print(CYAN + BOLD + f'{'='*55}' + RESET)

        # Per-column completeness detail
        if hasattr(self, 'per_col_missing'):
            print(BOLD + '\n  Missing % by column:' + RESET)
            for col, pct in self.per_col_missing.items():
                bar = '█' * int(pct / 5)  # visual bar: 1 block per 5%
                color = RED if pct > 10 else YELLOW if pct > 5 else GREEN
                print(f'    {col:<25} {color}{pct:>5.1f}%  {bar}{RESET}')

        return self

print('DataQualityReport class defined successfully.')

In [ ]:
# ============================================================
# RUN THE DATA QUALITY REPORT on the messy dataset
# ============================================================

# Define the same accuracy rules we used earlier (now packaged for the class)
rules = [
    {'rule_id': 'AMOUNT_POSITIVE',  'column': 'amount',   'check': lambda df: df['amount'] > 0},
    {'rule_id': 'AMOUNT_REALISTIC', 'column': 'amount',   'check': lambda df: df['amount'] <= 25_000},
    {'rule_id': 'AGE_VALID',        'column': 'user_age', 'check': lambda df: (df['user_age'] >= 18) & (df['user_age'] <= 110)},
    {'rule_id': 'FRAUD_BINARY',     'column': 'is_fraud', 'check': lambda df: df['is_fraud'].isin([0, 1])},
]

# Instantiate and run the quality report on the messy dataset
report_messy = DataQualityReport(
    df             = df_messy,
    name           = 'Messy Transaction Dataset (Before Cleaning)',
    timestamp_col  = 'timestamp_dt',    # use the parsed datetime column
    reference_date = datetime.datetime(2024, 1, 1),
    staleness_days = 365,
    accuracy_rules = rules
).compute()  # .compute() runs all four checks and returns self for chaining

report_messy.summary()  # print the colored report

In [ ]:
# ============================================================
# CLEAN THE DATA and run the report again for comparison
# ============================================================
# Demonstrate how quality score improves after fixing the problems we identified.

df_cleaned = df_messy.copy()

# Fix 1: Remove rows with negative amounts (accuracy violation — impossible values)
rows_before = len(df_cleaned)
df_cleaned = df_cleaned[~((df_cleaned['amount'] < 0) & df_cleaned['amount'].notna())]
print(f'Removed {rows_before - len(df_cleaned)} rows with negative amounts.')

# Fix 2: Cap extreme amounts at 25,000 (domain-informed upper bound)
n_capped = (df_cleaned['amount'] > 25_000).sum()
df_cleaned['amount'] = df_cleaned['amount'].clip(upper=25_000)
print(f'Capped {n_capped} extreme amount values to $25,000.')

# Fix 3: Set impossible ages to NaN (they're wrong — better to be missing than wrong)
invalid_age = ~((df_cleaned['user_age'] >= 18) & (df_cleaned['user_age'] <= 110))
n_bad_ages = (invalid_age & df_cleaned['user_age'].notna()).sum()
df_cleaned.loc[invalid_age, 'user_age'] = np.nan  # replace bad values with NaN
print(f'Nullified {n_bad_ages} impossible age values (will be imputed later).')

# Fix 4: Apply the category standardization mapping (consistency fix)
df_cleaned['merchant_category'] = df_cleaned['merchant_category'].map(
    category_mapping  # defined earlier: maps all variants → canonical form
)
print(f'Standardized merchant_category to canonical values.')

# Fix 5: Remove stale records (older than 365 days)
n_stale = df_cleaned['is_stale'].sum() if 'is_stale' in df_cleaned.columns else 0
if 'is_stale' in df_cleaned.columns:
    df_cleaned = df_cleaned[~df_cleaned['is_stale']]
print(f'Removed {n_stale} stale records (pre-2023 injected data).')

print(f'\nDataset size: {rows_before:,} → {len(df_cleaned):,} rows ({rows_before - len(df_cleaned):,} removed)')

# Run the quality report on cleaned data
report_cleaned = DataQualityReport(
    df             = df_cleaned,
    name           = 'Cleaned Transaction Dataset (After Cleaning)',
    timestamp_col  = 'timestamp_dt',
    reference_date = datetime.datetime(2024, 1, 1),
    staleness_days = 365,
    accuracy_rules = rules
).compute()

report_cleaned.summary()

In [ ]:
# ============================================================
# BEFORE vs AFTER VISUALIZATION: Quality Score Comparison
# ============================================================
# A radar chart (spider chart) is ideal for comparing multi-dimensional scores.

import numpy as np
import matplotlib.pyplot as plt

dimensions  = ['Completeness', 'Consistency', 'Accuracy', 'Timeliness']
scores_messy   = [report_messy.scores[d] for d in dimensions]
scores_cleaned = [report_cleaned.scores[d] for d in dimensions]

# --- Radar Chart ---
n_dims = len(dimensions)
# Create angles for each dimension (evenly spaced around a circle)
angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
# Close the polygon by repeating the first point
angles_closed = angles + [angles[0]]
scores_messy_closed   = scores_messy + [scores_messy[0]]
scores_cleaned_closed = scores_cleaned + [scores_cleaned[0]]

fig, (ax_radar, ax_bar) = plt.subplots(1, 2, figsize=(13, 5),
                                        subplot_kw={'projection': None})
# Recreate left axis as polar for radar chart
ax_radar.remove()
ax_radar = fig.add_subplot(121, projection='polar')
fig.suptitle('Before vs After Cleaning: Data Quality Score Comparison',
             fontsize=13, fontweight='bold')

# Plot messy (before) polygon
ax_radar.plot(angles_closed, scores_messy_closed, 'o-', linewidth=2,
              color='#E74C3C', label='Before cleaning')
ax_radar.fill(angles_closed, scores_messy_closed, alpha=0.15, color='#E74C3C')

# Plot cleaned (after) polygon
ax_radar.plot(angles_closed, scores_cleaned_closed, 'o-', linewidth=2,
              color='#27AE60', label='After cleaning')
ax_radar.fill(angles_closed, scores_cleaned_closed, alpha=0.15, color='#27AE60')

# Configure axis labels and range
ax_radar.set_xticks(angles)
ax_radar.set_xticklabels(dimensions, fontsize=10)
ax_radar.set_ylim(0, 105)
ax_radar.set_yticks([25, 50, 75, 90, 100])
ax_radar.set_yticklabels(['25', '50', '75', '90', '100'], fontsize=7)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax_radar.set_title('Quality Radar', fontsize=11, pad=18)

# --- Right: Bar chart of composite score before/after ---
ax_bar.clear()
bar_labels = ['Completeness', 'Consistency', 'Accuracy', 'Timeliness', 'COMPOSITE']
messy_vals   = [report_messy.scores[d] for d in dimensions]   + [report_messy.composite_score]
cleaned_vals = [report_cleaned.scores[d] for d in dimensions] + [report_cleaned.composite_score]

x = np.arange(len(bar_labels))
w = 0.35  # bar width
bars1 = ax_bar.bar(x - w/2, messy_vals,   w, label='Before', color='#E74C3C', alpha=0.8, edgecolor='white')
bars2 = ax_bar.bar(x + w/2, cleaned_vals, w, label='After',  color='#27AE60', alpha=0.8, edgecolor='white')

# Annotate bars with the score values
for bar in bars1:
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8, color='#E74C3C')
for bar in bars2:
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8, color='#27AE60')

ax_bar.axhline(90, color='gray', linestyle='--', lw=1, alpha=0.5, label='90% target')
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(bar_labels, fontsize=9)
ax_bar.set_ylabel('Quality Score (%)')
ax_bar.set_ylim(0, 115)
ax_bar.set_title('Dimension Scores: Before vs After', fontsize=11)
ax_bar.legend(fontsize=9)

# Highlight composite column
ax_bar.axvspan(3.6, 4.4, alpha=0.06, color='gray')

plt.tight_layout()
plt.show()

print(f'Composite score improvement: {report_messy.composite_score:.1f}% → {report_cleaned.composite_score:.1f}% '
      f'(+{report_cleaned.composite_score - report_messy.composite_score:.1f} percentage points)')

---

## Key Takeaways

| Dimension | What to Measure | Quick Code |
|---|---|---|
| **Completeness** | Missing % per column | `df.isnull().sum() / len(df) * 100` |
| **Consistency** | Unique values in categoricals | `df['col'].value_counts()` |
| **Accuracy** | Rule violations, z-score outliers | `(df['amount'] < 0).sum()` |
| **Timeliness** | % records within freshness window | `(ages <= threshold).mean() * 100` |

**The Cleaning Workflow:**
1. Measure quality BEFORE cleaning (so you know what you fixed)
2. Fix in order: accuracy first (wrong values corrupt statistics), then completeness, then consistency
3. Measure quality AFTER cleaning (document the improvement)
4. Never delete raw data — create a new cleaned version

**When to Accept vs Reject Low-Quality Data:**
- If completeness < 60% on a key column: investigate whether data is even collectible
- If >20% labels are noisy: consider re-labeling or programmatic correction
- If data is stale by > 2 years for a fast-moving domain: collect fresh data

---

---

## Self-Check Questions

Answer from memory first, then reveal.

---

**Question 1:** You have a dataset where 30% of `income` values are missing. Is this purely a completeness problem? What else might it indicate, and why does the reason for missingness matter?

<details>
<summary>Click to reveal answer</summary>

**It is a completeness problem — but the type of missingness (MCAR vs MAR vs MNAR) changes everything.**

If `income` is missing randomly (MCAR — completely at random), you can safely impute with the median and lose little information. However, in reality, high-income individuals are statistically less likely to self-report income (they value privacy, or the data comes from a tax-exempt source). This is **MNAR — Missing Not At Random**: whether the value is missing depends on the value itself.

**Why this matters:** If you impute MNAR data with the median, you systematically underestimate the income of wealthy users. For fraud detection, this is dangerous: high-income cards tend to have different fraud patterns (larger amounts, luxury merchants), so misrepresenting their income corrupts the model's behavior on that segment.

**The signal:** In MNAR, the missingness itself is informative — you should add a binary column `income_was_missing = 1/0` as a feature, so the model can learn that missing income has its own pattern.

</details>

---

**Question 2:** Your `gender` column contains: `'M'`, `'Male'`, `'m'`, `'MALE'`, `'male'`. Which data quality dimension is this, and what is the precise risk it creates for a fraud model?

<details>
<summary>Click to reveal answer</summary>

**This is a Consistency problem** (same concept stored differently).

**The risk for a fraud model:**
1. **If encoded as categorical:** One-hot encoding or label encoding treats 'M' and 'MALE' as completely different categories. A tree-based model learns separate fraud patterns for 'M', 'Male', 'm', etc. — even though they all mean the same thing. This dilutes statistical signal: instead of 3,000 examples of male users, you have 600 each across 5 meaningless variants.

2. **Distribution shift at inference time:** If the training data has mostly 'Male' but production data sends 'MALE', the model may have never seen 'MALE' in training and defaults to a zero-frequency category — producing unreliable predictions.

3. **SQL join failures:** If you need to join this table with another table on `gender`, the join silently drops rows where the strings don't exactly match (case-sensitive comparison).

**The fix:** Standardize before any encoding: `df['gender'] = df['gender'].str.lower().str.strip().map({'m': 'male', 'male': 'male', 'f': 'female', 'female': 'female'})`.

</details>

---

**Question 3:** A fraud model was trained in January and deployed in December. Transaction patterns shifted dramatically during summer (travel season, outdoor merchants). The model's precision dropped from 85% to 60%. Which quality dimension degraded, and what is the technical term for this phenomenon?

<details>
<summary>Click to reveal answer</summary>

**Timeliness degraded** — specifically through a phenomenon called **Concept Drift**.

**Concept drift** means the statistical relationship between input features and the target variable changes over time. In January, the model learned that high-amount gas station transactions late at night are fraud. But in summer, legitimate travel causes high-amount gas station transactions at unusual hours — the same pattern now means something different.

**Types of concept drift:**
- **Sudden drift**: A new fraud campaign launches overnight, instantly changing patterns
- **Gradual drift**: Spending patterns slowly evolve over months (what you're seeing here)
- **Seasonal drift**: Patterns repeat cyclically (summer travel, holiday shopping)
- **Recurring drift**: Old patterns return (back-to-school, tax season)

**Solutions:**
1. **Retrain regularly** (monthly retraining on a rolling 90-day window)
2. **Monitor model metrics** in production — a sudden drop in precision signals drift
3. **Drift detection algorithms** (ADWIN, Page-Hinkley test) that automatically trigger retraining
4. **Time-based features** (month, is_holiday) to help the model adapt to seasonal patterns

</details>

In [ ]:
# ============================================================
# BONUS: Concept Drift Visualization
# ============================================================
# Simulate how a fraud model's learned threshold drifts out of alignment
# when the underlying data distribution shifts over time.

np.random.seed(RANDOM_SEED)

# Simulate 12 months of average transaction amounts for two populations:
# legitimate (low, stable) and fraud (high, but DRIFTING over time)
months = np.arange(1, 13)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Legitimate spending mean: rises in Nov/Dec (holiday shopping)
legit_mean = 80 + 20 * np.sin((months - 3) * np.pi / 6) + np.random.normal(0, 3, 12)

# Fraud mean: starts high, drops as fraudsters adapt to detection, spikes in summer (travel)
fraud_mean = 250 - 30 * np.cos((months - 1) * np.pi / 6) + np.random.normal(0, 8, 12)

# Model threshold: trained on January data, stays FIXED at Jan's optimal threshold
jan_threshold = (legit_mean[0] + fraud_mean[0]) / 2  # midpoint = decision boundary in Jan

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle('Concept Drift: Why Timeliness Matters for Fraud Detection', fontsize=13, fontweight='bold')

# Top: Average amounts over time
ax1 = axes[0]
ax1.plot(months, legit_mean, 'o-', color='#4A90D9', linewidth=2, markersize=7, label='Avg. Legitimate Amount')
ax1.plot(months, fraud_mean, 's-', color='#E74C3C', linewidth=2, markersize=7, label='Avg. Fraud Amount')
ax1.axhline(jan_threshold, color='#2C3E50', linestyle='--', lw=2,
            label=f'Model threshold (fixed at Jan: ${jan_threshold:.0f})')
ax1.fill_between(months, legit_mean, fraud_mean, alpha=0.08, color='gray',
                 label='Separation gap (larger = easier to detect)')
ax1.set_xticks(months)
ax1.set_xticklabels(month_labels)
ax1.set_ylabel('Average Amount ($)')
ax1.set_title('Distribution Shift: Fraud & Legit Amounts Change, But Model Threshold Stays Fixed')
ax1.legend(fontsize=8, loc='lower right')

# Bottom: Simulated detection accuracy over time (proxy for precision)
ax2 = axes[1]
# Accuracy approximation: depends on how far apart the distributions are
gap = fraud_mean - legit_mean  # larger gap = easier separation = higher accuracy
# Normalize to a 60–95% accuracy range for visualization
accuracy = 60 + 35 * (gap - gap.min()) / (gap.max() - gap.min())
accuracy = np.clip(accuracy + np.random.normal(0, 2, 12), 55, 96)

ax2.bar(months, accuracy, color=[
    '#27AE60' if a >= 80 else '#F39C12' if a >= 70 else '#E74C3C' for a in accuracy
], edgecolor='white', linewidth=1)
ax2.axhline(80, color='gray', linestyle='--', lw=1.5, label='Acceptable accuracy threshold (80%)')
ax2.set_xticks(months)
ax2.set_xticklabels(month_labels)
ax2.set_ylabel('Simulated Precision (%)')
ax2.set_title('Model Precision Degrades as Drift Increases (green ≥ 80%, yellow 70–80%, red < 70%)')
ax2.set_ylim(50, 100)
ax2.legend(fontsize=8)

# Annotate a few key months
ax2.annotate('Model trained here', xy=(1, accuracy[0]), xytext=(1.5, accuracy[0] + 8),
             fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'), color='gray')

plt.tight_layout()
plt.show()

print('Key insight: Even a "perfect" January model degrades over time if not retrained.')
print('Solution: Retrain on a rolling window of recent data + monitor timeliness score.')

---

## What's Next?

You've now measured data quality across all four dimensions and built a reusable reporting tool. The next notebooks cover how to **fix** the problems you've just learned to detect:

- **Notebook 03** → Handling missing values (imputation strategies)
- **Notebook 04** → Outlier detection and treatment
- **Notebook 05** → Feature engineering (creating new, informative columns)
- **Notebook 06** → Encoding categorical variables
- **Notebook 07** → Scaling and normalization

**→ Continue to: `03_Handling_Missing_Values.ipynb`**

---
*Week 4 · Data Fundamentals & Preprocessing Pipelines*